# Deepfake Detector — Quick Inference
## Upload any face image → Get Real/Fake prediction + GradCAM heatmap

**How to use:**
1. Will take around 6 mins to install all the requirements
2. Use **Option A** to test a single image, or **Option B** to launch the interactive Gradio demo

**Requirements:** Trained model checkpoints at:  
`/content/drive/MyDrive/deepfake_detection/checkpoints/`

In [ ]:
# ============================================
# STEP 1: Setup (run once per session)
# ============================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
# Upgrade Pillow first to avoid the 'is_directory' ImportError
!pip install --upgrade Pillow
!pip install timm grad-cam gradio albumentations facenet-pytorch

In [ ]:
import importlib
import PIL
importlib.reload(PIL)
print(f'Pillow reloaded. Version: {PIL.__version__}')

Pillow reloaded. Version: 11.3.0


In [ ]:
# Force restart runtime to apply Pillow upgrade
import os
os._exit(0)

In [ ]:
# ============================================
# STEP 2: Load models (run once per session)
# ============================================

import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.cuda.amp import autocast
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from scipy.fft import dctn
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from facenet_pytorch import MTCNN

# FIX: ClassifierOutputTarget is more reliable across grad-cam versions
# for models with a single (B, 1) output logit.
try:
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRADCAM_TARGET = ClassifierOutputTarget(0)
except ImportError:
    from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget
    GRADCAM_TARGET = BinaryClassifierOutputTarget(1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT_DIR = '/content/drive/MyDrive/deepfake_detection/checkpoints'

# --- Model definitions ---
class ViTDeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('vit_large_patch14_clip_224', pretrained=False, num_classes=0)
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        return self.head(self.backbone(x))
    def extract_features(self, x):
        f = self.backbone(x)
        return self.head[1](self.head[0](f))

class FrequencyBranchDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=False, num_classes=0)
        old_conv = self.backbone.conv_stem
        new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                             stride=old_conv.stride, padding=old_conv.padding, bias=old_conv.bias is not None)
        self.backbone.conv_stem = new_conv
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        return self.head(self.backbone(x))
    def extract_features(self, x):
        f = self.backbone(x)
        return self.head[1](self.head[0](f))

class MetaClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1))
        self.temperature = nn.Parameter(torch.ones(1))
    def forward(self, x):
        return self.net(x)
    def predict_calibrated(self, x):
        return torch.sigmoid(self.net(x) / self.temperature)

# --- Load weights ---
print('Loading models... (this takes ~1-2 minutes)')

vit_model = ViTDeepfakeDetector().to(device)
vit_model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'vit_best.pth'), map_location=device, weights_only=False)['model_state_dict'])
vit_model.eval()
print('  ViT branch loaded')

freq_model = FrequencyBranchDetector().to(device)
freq_model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'freq_best.pth'), map_location=device, weights_only=False)['model_state_dict'])
freq_model.eval()
print('  Frequency branch loaded')

meta_model = MetaClassifier().to(device)
meta_model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'meta_calibrated.pth'), map_location=device, weights_only=False))
meta_model.eval()
print('  Meta-classifier loaded')

# --- Face detector (MTCNN) ---
# keep_all=False returns only the most confident face.
# min_face_size=40 filters out tiny noise detections.
face_detector = MTCNN(
    keep_all=False, min_face_size=40, thresholds=[0.6, 0.7, 0.7],
    device=device, post_process=False
)
print('  MTCNN face detector loaded')

# --- Setup GradCAM ---
# FIX: assertion so a wrong input size fails loudly instead of producing garbage
def reshape_transform(tensor, height=16, width=16):
    n_tokens = tensor.shape[1]
    expected = height * width + 1  # +1 for CLS
    assert n_tokens == expected, (
        f'Expected {expected} tokens (got {n_tokens}). '
        f'reshape_transform is hardcoded for 16x16 patches (ViT-L/14 @ 224).'
    )
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    return result.permute(0, 3, 1, 2)

cam = GradCAM(model=vit_model, target_layers=[vit_model.backbone.blocks[-1].norm1],
              reshape_transform=reshape_transform)

# --- Preprocessing ---
normalize = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

def compute_dct(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY).astype(np.float64)
    dct_coeffs = dctn(gray, norm='ortho')
    log_dct = np.log(1 + np.abs(dct_coeffs))
    mn, mx = log_dct.min(), log_dct.max()
    if mx - mn > 1e-8:
        log_dct = (log_dct - mn) / (mx - mn)
    else:
        log_dct = np.zeros_like(log_dct)
    return log_dct.astype(np.float32)

print('\nReady! Use Option A or Option B below to detect deepfakes.')


Loading models... (this takes ~1-2 minutes)
  ViT branch loaded
  Frequency branch loaded
  Meta-classifier loaded
  MTCNN face detector loaded

Ready! Use Option A or Option B below to detect deepfakes.


In [ ]:
# ============================================
# CORE PREDICTION FUNCTION
# ============================================

THRESHOLD = 0.3

def detect_and_crop_face(img, expand=0.30, min_size=60):
    """
    Detect the largest face in the image and return an expanded square crop.
    """
    h, w = img.shape[:2]

    try:
        pil_img = Image.fromarray(img)
        boxes, probs = face_detector.detect(pil_img)
    except Exception as e:
        print(f'  Face detector raised: {e}. Falling back to centre crop.')
        boxes = None

    if boxes is None or len(boxes) == 0:
        crop_size = min(h, w)
        top = (h - crop_size) // 2
        left = (w - crop_size) // 2
        return img[top:top+crop_size, left:left+crop_size], False

    x1, y1, x2, y2 = boxes[0]
    bw = x2 - x1
    bh = y2 - y1

    if max(bw, bh) < min_size:
        crop_size = min(h, w)
        top = (h - crop_size) // 2
        left = (w - crop_size) // 2
        return img[top:top+crop_size, left:left+crop_size], False

    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    side = max(bw, bh) * (1 + expand)

    half = side / 2
    x1e = int(max(0, cx - half))
    y1e = int(max(0, cy - half))
    x2e = int(min(w, cx + half))
    y2e = int(min(h, cy + half))

    face_crop = img[y1e:y2e, x1e:x2e]
    fh, fw = face_crop.shape[:2]
    side_final = min(fh, fw)
    if fh != fw:
        top = (fh - side_final) // 2
        left = (fw - side_final) // 2
        face_crop = face_crop[top:top+side_final, left:left+side_final]

    return face_crop, True


def detect_deepfake(input_image, use_face_detector=True):
    """
    Detect whether a face image is real or AI-generated using THRESHOLD.
    """
    if input_image is None:
        raise ValueError('detect_deepfake received None as input.')

    if isinstance(input_image, str):
        img = np.array(Image.open(input_image).convert('RGB'))
    elif isinstance(input_image, Image.Image):
        img = np.array(input_image.convert('RGB'))
    elif isinstance(input_image, np.ndarray):
        img = input_image
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        elif img.shape[2] == 4:
            img = img[:, :, :3]
    else:
        raise ValueError(f'Unsupported input type: {type(input_image)}')

    original_img = img.copy()

    if use_face_detector:
        face_crop, face_detected = detect_and_crop_face(img)
    else:
        h, w = img.shape[:2]
        crop_size = min(h, w)
        top = (h - crop_size) // 2
        left = (w - crop_size) // 2
        face_crop = img[top:top+crop_size, left:left+crop_size]
        face_detected = False

    img224 = np.array(Image.fromarray(face_crop).resize((224, 224), Image.LANCZOS))
    img_float = img224.astype(np.float32) / 255.0

    rgb_tensor = normalize(image=img224)['image'].unsqueeze(0).to(device)
    dct_array = compute_dct(img224)
    dct_tensor = torch.tensor(dct_array, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        with autocast():
            vit_backbone_out = vit_model.backbone(rgb_tensor)
            vit_logit = vit_model.head(vit_backbone_out).squeeze()
            vit_feat = vit_model.head[1](vit_model.head[0](vit_backbone_out)).float()
            vit_prob = torch.sigmoid(vit_logit).item()

            freq_backbone_out = freq_model.backbone(dct_tensor)
            freq_logit = freq_model.head(freq_backbone_out).squeeze()
            freq_feat = freq_model.head[1](freq_model.head[0](freq_backbone_out)).float()
            freq_prob = torch.sigmoid(freq_logit).item()

            fused_feat = torch.cat([vit_feat, freq_feat], dim=1)
            prob = meta_model.predict_calibrated(fused_feat).squeeze().item()

    targets = [GRADCAM_TARGET]
    grayscale_cam = cam(input_tensor=rgb_tensor, targets=targets)[0]
    overlay = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)

    label = 'AI-Generated (Fake)' if prob >= THRESHOLD else 'Real (Authentic)'
    confidence = prob if prob >= THRESHOLD else 1 - prob

    return {
        'label': label,
        'probability': prob,
        'confidence': confidence,
        'vit_prob': vit_prob,
        'freq_prob': freq_prob,
        'original': original_img,
        'cropped_face': img224,
        'gradcam': overlay,
        'face_detected': face_detected,
    }

print(f'detect_deepfake() ready. Current threshold: {THRESHOLD}')

detect_deepfake() ready. Current threshold: 0.3


---
## Option A: Test a Single Image

Upload an image to Colab or provide a path from Google Drive.

In [ ]:
# --- OPTION A1: Upload from your computer ---
from google.colab import files

print('Select an image to test:')
uploaded = files.upload()

for filename in uploaded:
    result = detect_deepfake(filename)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(result['original'])
    axes[0].set_title('Uploaded Image', fontsize=12)
    axes[0].axis('off')

    face_title = 'Detected Face (fed to model)' if result['face_detected'] else 'Centre Crop (no face detected)'
    axes[1].imshow(result['cropped_face'])
    axes[1].set_title(face_title, fontsize=12)
    axes[1].axis('off')

    axes[2].imshow(result['gradcam'])
    axes[2].set_title('GradCAM Heatmap', fontsize=12)
    axes[2].axis('off')

    # Color based on updated THRESHOLD
    color = '#F44336' if result['probability'] >= THRESHOLD else '#4CAF50'
    plt.suptitle(
        f"{result['label']}  |  Confidence: {result['confidence']:.1%}\n"
        f"P(fake): {result['probability']:.4f}  |  "
        f"ViT: {result['vit_prob']:.3f}  |  Freq: {result['freq_prob']:.3f}",
        fontsize=14, color=color, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# --- OPTION A2: Test from a Google Drive path ---
IMAGE_PATH = '/content/drive/MyDrive/test_image.jpg'  # CHANGE THIS

if os.path.exists(IMAGE_PATH):
    result = detect_deepfake(IMAGE_PATH)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(result['original'])
    axes[0].set_title('Uploaded Image', fontsize=12)
    axes[0].axis('off')

    face_title = 'Detected Face (fed to model)' if result['face_detected'] else 'Centre Crop (no face detected)'
    axes[1].imshow(result['cropped_face'])
    axes[1].set_title(face_title, fontsize=12)
    axes[1].axis('off')

    axes[2].imshow(result['gradcam'])
    axes[2].set_title('GradCAM Heatmap', fontsize=12)
    axes[2].axis('off')

    color = '#F44336' if result['probability'] >= 0.5 else '#4CAF50'
    plt.suptitle(
        f"{result['label']}  |  Confidence: {result['confidence']:.1%}\n"
        f"P(fake): {result['probability']:.4f}  |  "
        f"ViT: {result['vit_prob']:.3f}  |  Freq: {result['freq_prob']:.3f}",
        fontsize=14, color=color, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()
else:
    print(f'File not found: {IMAGE_PATH}')
    print('Update IMAGE_PATH to point to your image.')


In [ ]:
# --- OPTION A3: Test multiple images from a folder ---
FOLDER_PATH = '/content/drive/MyDrive/test_faces/'  # CHANGE THIS

if os.path.exists(FOLDER_PATH):
    image_files = [f for f in os.listdir(FOLDER_PATH)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]

    print(f'Found {len(image_files)} images in {FOLDER_PATH}\n')

    for filename in sorted(image_files):
        path = os.path.join(FOLDER_PATH, filename)
        result = detect_deepfake(path)

        status = 'FAKE' if result['probability'] >= 0.5 else 'REAL'
        face_tag = '[face]' if result['face_detected'] else '[ctr ]'
        print(f'{status:<4s} {face_tag}  {filename:<35s}  P(fake)={result["probability"]:.4f}  '
              f'Confidence={result["confidence"]:.1%}  '
              f'(ViT={result["vit_prob"]:.3f}, Freq={result["freq_prob"]:.3f})')
else:
    print(f'Folder not found: {FOLDER_PATH}')
    print('Update FOLDER_PATH or skip this cell.')


---
## Option B: Interactive Gradio Demo

Launch a web interface — drag and drop images to test. Gets a shareable public URL.

In [ ]:
import gradio as gr

def gradio_predict(input_image):
    if input_image is None:
        return 'No image uploaded', None, None

    result = detect_deepfake(input_image)

    # Color based on updated THRESHOLD
    color = '#F44336' if result['probability'] >= THRESHOLD else '#4CAF50'
    face_note = (
        "Face detected and cropped by MTCNN."
        if result['face_detected']
        else "**No face detected** - fell back to centre crop. Result may be unreliable."
    )

    text = (
        f"### Result: <span style='color:{color}'>{result['label']}</span>\n"
        f"**Confidence:** {result['confidence']:.1%}\n"
        f"**Probability Score:** {result['probability']:.4f}\n\n"
        f"---\n"
        f"**Analysis Breakdown:**\n"
        f"*   **Semantic (ViT):** {result['vit_prob']:.4f}\n"
        f"*   **Frequency (DCT):** {result['freq_prob']:.4f}\n\n"
        f"---\n"
        f"*{face_note}*"
    )

    return text, result['cropped_face'], result['gradcam']

with gr.Blocks(theme=gr.themes.Soft(), title='Deepfake Detector') as demo:
    gr.Markdown('# AI-Generated Face Detector')
    gr.Markdown(
        'Upload any photo. The system will detect '
        'the face, crop it, and analyse authenticity.'
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(label='Upload Image', type='numpy')
            submit_btn = gr.Button('Run Detection', variant='primary')

        with gr.Column(scale=1):
            output_text = gr.Markdown(label='Prediction Details')
            with gr.Row():
                output_face = gr.Image(label='Detected Face (analysed)')
                output_cam = gr.Image(label='GradCAM Heatmap')

    submit_btn.click(
        fn=gradio_predict,
        inputs=input_img,
        outputs=[output_text, output_face, output_cam]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://75ff32f2f9b1d10447.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
